In [1]:
import pandas as pd
import numpy as np
import warnings

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_columns', 500)
pd.set_option('display.max_rows', 500)
pd.set_option('display.float_format', lambda x: '%.4f' % x)
warnings.filterwarnings('ignore')

In [2]:
import pandas as pd
import numpy as np
import pymysql
import psycopg2
import re
import os
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'Malgun Gothic'
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 경로 설정 ──────────────────────────────────────────────────────
PACER_DIR       = './csv_2016'
PATENTSVIEW_DIR = './patentsview'
DATA_DIR        = './data'
os.makedirs(DATA_DIR, exist_ok=True)

# ── DB 접속 정보 ───────────────────────────────────────────────────
DB_149_HOST  = '192.168.1.149'
DB_116_HOST  = '192.168.1.116'
DB_USER      = 'root'
DB_PW        = 'doslvkdlqm!'
PG_USER      = 'postgres'
PG_PW        = 'doslvkdlqm'
RANDOM_STATE = 42

# ── 21개 피처 목록 (논문 기준) ────────────────────────────────────
FEATURE_COLS = [
    'MCD', 'SCD',
    'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H',
    'CS', 'ABS', 'CTS', 'CTC', 'PTS',
    'PK', 'SK', 'PPS',
    'CD', 'IC', 'IE', 'FIE', 'ET',
    'TCP',
    'LI1', 'LI3', 'LI5', 'LI7'
]

print('환경 설정 완료')
print(f'피처 수 (LI 포함 28개): {len(FEATURE_COLS)}')

환경 설정 완료
피처 수 (LI 포함 28개): 28


#### Step 1. PACER 데이터 로딩 & 소송특허 추출

In [3]:
cases   = pd.read_csv(f'{PACER_DIR}/cases.csv',   low_memory=False)
patents = pd.read_csv(f'{PACER_DIR}/patents.csv',  low_memory=False)
names   = pd.read_csv(f'{PACER_DIR}/names.csv',    low_memory=False)

# 침해소송 필터
inf_cases = cases[cases['case_type_1'] == 1.0].copy()
inf_cases['date_filed'] = pd.to_datetime(inf_cases['date_filed'], errors='coerce')

# Patent 타입 필터
lit_patents = patents[
    (patents['case_row_id'].isin(inf_cases['case_row_id'])) &
    (patents['patent_doc_type'].str.strip() == 'Patent')
].copy()
lit_patents['patent'] = lit_patents['patent'].astype(str).str.strip()

def normalize_patent(s):
    """특허번호 정규화: 대문자 + prefix 유지 + 앞 0 제거"""
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    m = re.match(r'^([A-Z]*)(\d+)$', s)
    if not m:
        return None
    prefix, digits = m.group(1), m.group(2)
    return prefix + digits.lstrip('0')

lit_patents['patent_norm'] = lit_patents['patent'].apply(normalize_patent)
litigated_set      = set(lit_patents['patent'].dropna().unique())
litigated_norm_set = set(lit_patents['patent_norm'].dropna().unique())
litigated_set_int  = {int(p) for p in litigated_set if p.isdigit()}

print(f'침해소송 케이스: {len(inf_cases):,}건')
print(f'소송 특허 unique: {len(litigated_set):,}건')
print(f'  └ 정수 변환 가능 (PatentsView용): {len(litigated_set_int):,}건')

침해소송 케이스: 46,907건
소송 특허 unique: 38,214건
  └ 정수 변환 가능 (PatentsView용): 33,657건


#### Step 2. LI 피처 (Litigation Intensity)

In [4]:
# 원고 추출
plaintiffs = (
    names[names['party_type'].str.strip().str.lower() == 'plaintiff']
    [['case_row_id', 'name']]
    .drop_duplicates('case_row_id')
    .rename(columns={'name': 'plaintiff'})
)
case_lit = (
    inf_cases[['case_row_id', 'date_filed']]
    .merge(plaintiffs, on='case_row_id', how='left')
    .dropna(subset=['date_filed', 'plaintiff'])
    .sort_values('date_filed')
    .reset_index(drop=True)
)

def calc_li_all(case_lit_df):
    df = case_lit_df.copy()
    dates      = df['date_filed'].values.astype('datetime64[D]')
    pltfs      = df['plaintiff'].values
    for w in [1, 3, 5, 7]:
        window = np.timedelta64(w * 365, 'D')
        col = np.zeros(len(df), dtype=np.float32)
        for i in range(len(df)):
            mask = (pltfs == pltfs[i]) & (dates >= dates[i] - window) & (dates < dates[i])
            col[i] = mask.sum()
        df[f'LI{w}'] = col
        print(f'  LI{w} 완료')
    return df

print('LI 계산 중...')
case_lit = calc_li_all(case_lit)

patent_li = (
    lit_patents[['case_row_id', 'patent']]
    .merge(case_lit[['case_row_id', 'LI1', 'LI3', 'LI5', 'LI7']], on='case_row_id', how='left')
    .groupby('patent')[['LI1', 'LI3', 'LI5', 'LI7']]
    .mean()
    .reset_index()
)
patent_li['patent_id'] = pd.to_numeric(patent_li['patent'], errors='coerce')

print(f'LI 완료: {len(patent_li):,}건')
print(patent_li[['LI1','LI3','LI5','LI7']].describe().round(2))

LI 계산 중...
  LI1 완료
  LI3 완료
  LI5 완료
  LI7 완료
LI 완료: 38,214건
             LI1        LI3        LI5        LI7
count 37837.0000 37837.0000 37837.0000 37837.0000
mean      0.7600     1.4100     1.7400     1.9200
std       2.5400     4.5100     5.9300     6.3500
min       0.0000     0.0000     0.0000     0.0000
25%       0.0000     0.0000     0.0000     0.0000
50%       0.0000     0.0000     0.0000     0.0000
75%       0.5000     1.0000     1.0000     1.2500
max      67.5800   134.5000   138.1700   140.8500


#### Step 3. NL 샘플링
비소송 특허를 g_patent.tsv에서 샘플링. 소송 특허 대비 1:5 비율.

In [5]:
all_pid = pd.read_csv(
    f'{PATENTSVIEW_DIR}/g_patent.tsv', sep='\t',
    usecols=['patent_id', 'patent_type', 'num_claims', 'withdrawn'],
    low_memory=False
)
all_pid['patent_id'] = pd.to_numeric(all_pid['patent_id'], errors='coerce')
all_pid = all_pid.dropna(subset=['patent_id']).astype({'patent_id': int})

# utility 특허 + 취하 안 된 것만 NL 후보로 사용
# design/plant 특허는 소송 특허와 성질이 달라 NL에 섞이면 모델이 타입 차이를 학습할 수 있음
all_pid_util = all_pid[
    (all_pid['patent_type'] == 'utility') &
    (all_pid['withdrawn'] == 0)
]

print(f'전체 PatentsView 특허:  {len(all_pid):,}건')
print(f'  └ utility + 비취하:   {len(all_pid_util):,}건')

nl_pool    = set(all_pid_util['patent_id']) - litigated_set_int
nl_n       = len(litigated_set_int) * 5
nl_set_int = set(
    pd.Series(list(nl_pool)).sample(n=nl_n, random_state=RANDOM_STATE)
)
print(f'NL 풀: {len(nl_pool):,}건  →  샘플: {len(nl_set_int):,}건')

# num_claims → Step 5 이후 CTS 결측 보완에 사용 (전체 특허 커버)
patent_numclaims = (
    all_pid[['patent_id', 'num_claims']]
    .dropna(subset=['num_claims'])
    .astype({'num_claims': int})
)
print(f'num_claims 보유: {len(patent_numclaims):,}건')

전체 PatentsView 특허:  8,532,000건
  └ utility + 비취하:   8,512,829건
NL 풀: 8,479,201건  →  샘플: 168,285건
num_claims 보유: 8,532,000건


#### Step 4. PatentsView 기본 피처 추출
PK, SK, PPS, ET, IE, FIE, ABS, MCD, SCD, TD

In [49]:
def read_pv_chunks(filepath, usecols, target_set, filter_col='patent_id', chunksize=500_000):
    """PatentsView TSV 청크 로딩 — target_set 필터링"""
    chunks = []
    for chunk in pd.read_csv(filepath, sep='\t', usecols=usecols,
                              low_memory=False, chunksize=chunksize):
        chunk[filter_col] = pd.to_numeric(chunk[filter_col], errors='coerce')
        chunk = chunk[chunk[filter_col].isin(target_set)]
        if len(chunk) > 0:
            chunks.append(chunk)
    return pd.concat(chunks) if chunks else pd.DataFrame(columns=usecols)

In [23]:
def read_pv_chunks(filepath, usecols, target_set, filter_col='patent_id', chunksize=500_000):
    """PatentsView TSV 청크 로딩 — target_set 필터링"""
    chunks = []
    for chunk in pd.read_csv(filepath, sep='\t', usecols=usecols,
                              low_memory=False, chunksize=chunksize):
        chunk[filter_col] = pd.to_numeric(chunk[filter_col], errors='coerce')
        chunk = chunk[chunk[filter_col].isin(target_set)]
        if len(chunk) > 0:
            chunks.append(chunk)
    return pd.concat(chunks) if chunks else pd.DataFrame(columns=usecols)


def extract_pv_features(target_set, label=''):
    """PatentsView 기본 피처 추출: PK, SK, PPS, ET, IE, FIE, ABS, IPC"""
    print(f'[{label}] PatentsView 피처 추출 ({len(target_set):,}건)')

    # PK
    print('  PK...', end=' ', flush=True)
    pk = (read_pv_chunks(f'{PATENTSVIEW_DIR}/g_us_patent_citation.tsv',
                         ['patent_id', 'citation_sequence'], target_set)
          .groupby('patent_id').size().reset_index().rename(columns={0: 'PK'}))
    print(f'{len(pk):,}건')

    # SK
    print('  SK...', end=' ', flush=True)
    sk = (read_pv_chunks(f'{PATENTSVIEW_DIR}/g_other_reference.tsv',
                         ['patent_id', 'other_reference_sequence'], target_set)
          .groupby('patent_id').size().reset_index().rename(columns={0: 'SK'}))
    print(f'{len(sk):,}건')

    # PPS
    print('  PPS...', end=' ', flush=True)
    pps = (read_pv_chunks(f'{PATENTSVIEW_DIR}/g_foreign_priority.tsv',
                          ['patent_id', 'foreign_country_filed'], target_set)
           .groupby('patent_id')['foreign_country_filed'].nunique()
           .reset_index().rename(columns={'foreign_country_filed': 'PPS'}))
    print(f'{len(pps):,}건')

    # ET
    print('  ET...', end=' ', flush=True)
    pat_df = read_pv_chunks(f'{PATENTSVIEW_DIR}/g_patent.tsv',
                            ['patent_id', 'patent_date'], target_set)
    app_df = read_pv_chunks(f'{PATENTSVIEW_DIR}/g_application.tsv',
                            ['patent_id', 'filing_date'], target_set)
    et_df  = pat_df.merge(app_df, on='patent_id', how='inner')
    et_df['ET'] = (pd.to_datetime(et_df['patent_date'], errors='coerce') -
                   pd.to_datetime(et_df['filing_date'], errors='coerce')).dt.days
    et = et_df[['patent_id', 'ET']].dropna()
    print(f'{len(et):,}건')

    # IE / FIE
    print('  IE/FIE...', end=' ', flush=True)
    inv_df = read_pv_chunks(
        f'{PATENTSVIEW_DIR}/g_inventor_not_disambiguated.tsv',
        ['patent_id', 'inventor_sequence', 'rawlocation_id'], target_set
    )
    ie = (inv_df.groupby('patent_id')['inventor_sequence']
          .count().reset_index().rename(columns={'inventor_sequence': 'IE'}))
    need_locs = set(inv_df['rawlocation_id'].dropna().unique())
    loc_chunks = []
    for chunk in pd.read_csv(
        f'{PATENTSVIEW_DIR}/g_location_not_disambiguated.tsv', sep='\t',
        usecols=['rawlocation_id', 'raw_country'], low_memory=False, chunksize=500_000
    ):
        loc_chunks.append(chunk[chunk['rawlocation_id'].isin(need_locs)])
    loc_df = pd.concat(loc_chunks).drop_duplicates('rawlocation_id')
    fie = (inv_df.merge(loc_df, on='rawlocation_id', how='left')
           .groupby('patent_id')['raw_country'].nunique()
           .reset_index().rename(columns={'raw_country': 'FIE'}))
    print(f'IE {len(ie):,}건 / FIE {len(fie):,}건')

    # ABS
    print('  ABS...', end=' ', flush=True)
    abs_df = read_pv_chunks(f'{PATENTSVIEW_DIR}/g_patent_abstract.tsv',
                            ['patent_id', 'patent_abstract'], target_set)
    abs_df['ABS'] = abs_df['patent_abstract'].fillna('').apply(lambda x: len(str(x).split()))
    abs_ = abs_df[['patent_id', 'ABS']]
    print(f'{len(abs_):,}건')

    # MCD / SCD / TD
    print('  IPC...', end=' ', flush=True)
    ipc_df = read_pv_chunks(
        f'{PATENTSVIEW_DIR}/g_ipc_at_issue.tsv',
        ['patent_id', 'section', 'ipc_class', 'subclass'], target_set
    )
    ipc_df['main_class'] = (ipc_df['section'].fillna('').astype(str) +
                            ipc_df['ipc_class'].fillna(0).astype(str))
    ipc_df['sub_class']  = ipc_df['main_class'] + ipc_df['subclass'].fillna('').astype(str)
    ipc_rows = []
    for pid, grp in ipc_df.groupby('patent_id'):
        row = {'patent_id': pid,
               'MCD': grp['main_class'].nunique(),
               'SCD': grp['sub_class'].nunique()}
        secs = set(grp['section'].dropna().astype(str).str.upper())
        for s in 'ABCDEFGH':
            row[f'TD_{s}'] = int(s in secs)
        ipc_rows.append(row)
    ipc = pd.DataFrame(ipc_rows)
    print(f'{len(ipc):,}건')

    # 통합
    feat = pd.DataFrame({'patent_id': list(target_set)})
    for df_, col in [(pk,'PK'),(sk,'SK'),(pps,'PPS'),(et,'ET'),(ie,'IE'),(fie,'FIE'),(abs_,'ABS')]:
        feat = feat.merge(df_, on='patent_id', how='left')
    feat = feat.merge(ipc, on='patent_id', how='left')
    print(f'  → 통합 완료: {len(feat):,}건\n')
    return feat

print('피처 추출 함수 준비 완료')

피처 추출 함수 준비 완료


In [24]:
pv_L  = extract_pv_features(litigated_set_int, label='L')
pv_NL = extract_pv_features(nl_set_int,        label='NL')

[L] PatentsView 피처 추출 (33,657건)
  PK... 33,198건
  SK... 18,875건
  PPS... 4,429건
  ET... 33,630건
  IE/FIE... IE 33,629건 / FIE 33,629건
  ABS... 33,630건
  IPC... 33,629건
  → 통합 완료: 33,657건

[NL] PatentsView 피처 추출 (168,285건)
  PK... 153,591건
  SK... 88,144건
  PPS... 65,521건
  ET... 168,284건
  IE/FIE... IE 168,276건 / FIE 168,276건
  ABS... 168,285건
  IPC... 168,278건
  → 통합 완료: 168,285건



In [26]:
# PatentsView 기본 피처 중간 저장
pv_L.to_pickle(f'{DATA_DIR}/pv_L.pkl')
pv_NL.to_pickle(f'{DATA_DIR}/pv_NL.pkl')

print('저장 완료')
print(f'  {DATA_DIR}/pv_L.pkl   — {len(pv_L):,}건 / {len(pv_L.columns)}컬럼')
print(f'  {DATA_DIR}/pv_NL.pkl  — {len(pv_NL):,}건 / {len(pv_NL.columns)}컬럼')
print(f'\n컬럼 목록: {pv_L.columns.tolist()}')

저장 완료
  ./data/pv_L.pkl   — 33,657건 / 18컬럼
  ./data/pv_NL.pkl  — 168,285건 / 18컬럼

컬럼 목록: ['patent_id', 'PK', 'SK', 'PPS', 'ET', 'IE', 'FIE', 'ABS', 'MCD', 'SCD', 'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H']


#### Step 5. [NEW] 청구항 피처 — CTS / PTS / CTC
**CTS** (Claim Total count): 특허당 총 청구항 수  
**PTS** (Principal/independent claim count): 독립항 수 (`dependent` 컬럼이 NaN인 행)  
**CTC** (Claim Text Complexity): 청구항 전체 텍스트 단어 수 합계  

데이터 소스: `./patentsview/g_claims_2016.tsv` (2016년 등록 특허 334,674건)

In [27]:
# 소송 특허 등록 연도 분포 확인
import pandas as pd
pat = pd.read_csv('./patentsview/g_patent.tsv', sep='\t',
                usecols=['patent_id', 'patent_date'], low_memory=False)
pat['patent_id'] = pd.to_numeric(pat['patent_id'], errors='coerce')
pat = pat[pat['patent_id'].isin(litigated_set_int)]
pat['year'] = pd.to_datetime(pat['patent_date']).dt.year
print(pat['year'].value_counts().sort_index())


year
1976       7
1977       7
1978       7
1979       7
1980      12
1981      17
1982      13
1983      23
1984      24
1985      50
1986      79
1987     151
1988     155
1989     257
1990     293
1991     351
1992     419
1993     452
1994     579
1995     691
1996     885
1997    1032
1998    1482
1999    1637
2000    1852
2001    2036
2002    1977
2003    2107
2004    1870
2005    1646
2006    1876
2007    1529
2008    1279
2009    1273
2010    1532
2011    1410
2012    1320
2013    1258
2014     998
2015     686
2016     327
2017      21
2018       3
Name: count, dtype: int64


In [29]:
print('g_claims_2016.tsv 로딩...', flush=True)
claims_2016 = pd.read_csv(
    f'{PATENTSVIEW_DIR}/g_claims_2016.tsv', sep='\t',
    usecols=['patent_id', 'claim_sequence', 'claim_text', 'dependent'],
    low_memory=False
)
claims_2016['patent_id'] = pd.to_numeric(claims_2016['patent_id'], errors='coerce')
claims_2016 = claims_2016.dropna(subset=['patent_id']).astype({'patent_id': int})

print(f'총 행 수: {len(claims_2016):,}')
print(f'unique 특허 수: {claims_2016["patent_id"].nunique():,}')
print(f'dependent NaN 비율 (= 독립항): {claims_2016["dependent"].isna().mean():.1%}')
print()
print(claims_2016.head(3))

g_claims_2016.tsv 로딩...
총 행 수: 4,947,437
unique 특허 수: 304,126
dependent NaN 비율 (= 독립항): 15.3%

   patent_id  claim_sequence  \
0    9511332               2   
1    9364641               2   
2    9327125               6   

                                                                                                                                                         claim_text  \
0  3. The stirring apparatus of claim 1 , wherein said deflector has a continuous peripheral rim configured for frictional contact with a bottom of said container.   
1                                                                            3. The method of claim 2 , wherein holding and moving are effected by the same person.   
2                                                                                     7. The method of claim 6 wherein the frequency range is from 3 kHz to 20 kHz.   

  dependent  
0   claim 1  
1   claim 2  
2   claim 6  


In [31]:
# ── CTS: 총 청구항 수 ──────────────────────────────────────────────
cts_df = (
    claims_2016.groupby('patent_id').size()
    .reset_index().rename(columns={0: 'CTS'})
)

# ── PTS: 독립항 수 ────────────────────────────────────────────────
pts_df = (
    claims_2016[claims_2016['dependent'].isna()]
    .groupby('patent_id').size()
    .reset_index().rename(columns={0: 'PTS'})
)

# ── CTC: 청구항 텍스트 총 단어 수 ─────────────────────────────────
claims_2016['word_count'] = (
    claims_2016['claim_text'].fillna('').apply(lambda x: len(str(x).split()))
)
ctc_df = (
    claims_2016.groupby('patent_id')['word_count'].sum()
    .reset_index().rename(columns={'word_count': 'CTC'})
)

print(f'CTS 계산 완료: {len(cts_df):,}건')
print(f'PTS 계산 완료: {len(pts_df):,}건')
print(f'CTC 계산 완료: {len(ctc_df):,}건')
print()

# ── 통합 ──────────────────────────────────────────────────────────
claims_feat = cts_df.merge(pts_df, on='patent_id', how='left').merge(ctc_df, on='patent_id', how='left')

print('=== CTS / PTS / CTC 기술통계 ===')
print(claims_feat[['CTS','PTS','CTC']].describe().round(1))

CTS 계산 완료: 304,126건
PTS 계산 완료: 304,126건
CTC 계산 완료: 304,126건

=== CTS / PTS / CTC 기술통계 ===
              CTS         PTS         CTC
count 304126.0000 304126.0000 304126.0000
mean      16.3000      2.5000   1072.0000
std        8.6000      1.4000    693.3000
min        1.0000      1.0000      3.0000
25%       10.0000      1.0000    635.0000
50%       17.0000      2.0000    954.0000
75%       20.0000      3.0000   1343.0000
max      310.0000    100.0000  42642.0000


In [35]:
# PatentsView 기본 피처 + 청구항 피처 합쳐서 중간 저장
# 나머지 연도 g_claims_YYYY.tsv 확보 후 claims_feat에 추가하고 재저장 예정

pv_L  = pv_L.merge(claims_feat, on='patent_id', how='left')
pv_NL = pv_NL.merge(claims_feat, on='patent_id', how='left')

pv_L.to_pickle(f'{DATA_DIR}/pv_L.pkl')
pv_NL.to_pickle(f'{DATA_DIR}/pv_NL.pkl')

print('저장 완료')
print(f'  pv_L  : {len(pv_L):,}건 / 컬럼 {len(pv_L.columns)}개 {pv_L.columns.tolist()}')
print(f'  pv_NL : {len(pv_NL):,}건 / 컬럼 {len(pv_NL.columns)}개')
print()
print('CTS 결측률')
print(f'  L  : {pv_L["CTS"].isna().mean():.1%}')
print(f'  NL : {pv_NL["CTS"].isna().mean():.1%}')

저장 완료
  pv_L  : 33,657건 / 컬럼 21개 ['patent_id', 'PK', 'SK', 'PPS', 'ET', 'IE', 'FIE', 'ABS', 'MCD', 'SCD', 'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H', 'CTS', 'PTS', 'CTC']
  pv_NL : 168,285건 / 컬럼 21개

CTS 결측률
  L  : 99.0%
  NL : 96.4%


In [ ]:
import os

target = f'{PATENTSVIEW_DIR}/g_claims_2016.tsv'
if os.path.exists(target):
    size_gb = os.path.getsize(target) / 1e9
    os.remove(target)
    print(f'삭제 완료: {target}  ({size_gb:.2f} GB 확보)')
else:
    print(f'파일 없음 (이미 삭제됨): {target}')

---
## ▶ 커널 재시작 후 여기서부터 재개

커널 재시작 시 아래 순서로 실행:
1. **cell 1** — imports & config
2. **cell 3** — PACER 로딩 (`litigated_set_int`, `lit_patents`, `inf_cases`, `names`)
3. **cell 5** — LI 계산 (`patent_li`)
4. **cell 7** — NL 샘플링 (`nl_set_int`, `patent_numclaims`)
5. **아래 체크포인트 셀** — `pv_L`, `pv_NL` 로드 (Step 4~5 전체 대체)
6. **Step 6 이후** 계속 진행

In [9]:
# 체크포인트 로드 — Step 4~5 대체 (PatentsView 추출 + 청구항 피처 포함)
pv_L  = pd.read_pickle(f'{DATA_DIR}/pv_L.pkl')
pv_NL = pd.read_pickle(f'{DATA_DIR}/pv_NL.pkl')

print('로드 완료')
print(f'  pv_L  : {len(pv_L):,}건 / 컬럼 {len(pv_L.columns)}개')
print(f'  pv_NL : {len(pv_NL):,}건 / 컬럼 {len(pv_NL.columns)}개')
print(f'  컬럼 목록: {pv_L.columns.tolist()}')
print()
print('현재 CTS/PTS/CTC 결측률 (2004~2015 파일 처리 전)')
for col in ['CTS', 'PTS', 'CTC']:
    if col in pv_L.columns:
        print(f'  L  {col}: {pv_L[col].isna().mean():.1%}')
        print(f'  NL {col}: {pv_NL[col].isna().mean():.1%}')

로드 완료
  pv_L  : 33,657건 / 컬럼 21개
  pv_NL : 168,285건 / 컬럼 21개
  컬럼 목록: ['patent_id', 'PK', 'SK', 'PPS', 'ET', 'IE', 'FIE', 'ABS', 'MCD', 'SCD', 'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H', 'CTS', 'PTS', 'CTC']

현재 CTS/PTS/CTC 결측률 (2004~2015 파일 처리 전)
  L  CTS: 28.7%
  NL CTS: 45.7%
  L  PTS: 28.7%
  NL PTS: 45.7%
  L  CTC: 28.7%
  NL CTC: 45.7%


### Step 5-C. 다년도 청구항 피처 보완 — g_claims_2004~2015

`g_claims_2016.tsv` + `num_claims` 보완으로 CTS는 대부분 채워졌지만,  
**PTS(독립항 수)·CTC(텍스트 단어수)는 2016년 특허에만 존재**.  
새로 다운로드한 2004~2015 연도 파일로 나머지 연도 특허의 CTS/PTS/CTC를 전부 채웁니다.

| 처리 전 | 처리 후 |
|---------|--------|
| CTS: 2016 파일 + num_claims (100% 가까이) | CTS: 실제 청구항 데이터로 교체 (2004~2016 연도 커버) |
| PTS: 2016 특허만 (~3%) | PTS: 2004~2016 특허 (grant year 기준 커버 대폭 향상) |
| CTC: 2016 특허만 (~3%) | CTC: 2004~2016 특허 (동일) |

In [12]:
import gc

def compute_claims_feat(filename):
    """g_claims_YYYY.tsv에서 CTS/PTS/CTC 계산"""
    path = f'{PATENTSVIEW_DIR}/{filename}'
    if not os.path.exists(path):
        print(f'    없음: {filename}')
        return pd.DataFrame(columns=['patent_id', 'CTS', 'PTS', 'CTC'])

    df = pd.read_csv(path, sep='\t',
                     usecols=['patent_id', 'claim_sequence', 'claim_text', 'dependent'],
                     low_memory=False)
    df['patent_id'] = pd.to_numeric(df['patent_id'], errors='coerce')
    df = df.dropna(subset=['patent_id']).astype({'patent_id': int})

    cts = df.groupby('patent_id').size().reset_index().rename(columns={0: 'CTS'})
    pts = (df[df['dependent'].isna()]
           .groupby('patent_id').size().reset_index().rename(columns={0: 'PTS'}))
    df['word_count'] = df['claim_text'].fillna('').apply(lambda x: len(str(x).split()))
    ctc = (df.groupby('patent_id')['word_count'].sum()
           .reset_index().rename(columns={'word_count': 'CTC'}))

    result = cts.merge(pts, on='patent_id', how='left').merge(ctc, on='patent_id', how='left')
    del df
    gc.collect()
    return result


# ── 2004~2015 연도별 처리 ──────────────────────────────────────────
years = range(1999, 2004)
all_new = []

for year in years:
    fname = f'g_claims_{year}.tsv'
    print(f'{year}년 처리 중...', end=' ', flush=True)
    feat = compute_claims_feat(fname)
    if len(feat) > 0:
        print(f'{len(feat):,}건')
        all_new.append(feat)
    else:
        print('건너뜀')

claims_new = pd.concat(all_new, ignore_index=True)
print(f'\n2004-2015 통합: {len(claims_new):,}건 unique 특허')


# ── pv_L / pv_NL 업데이트 ──────────────────────────────────────────
# 현재 상태: 2016 파일(CTS+PTS+CTC 완전) + num_claims 보완(CTS만, PTS/CTC=NaN)
# 업데이트: 해당 연도 파일로 CTS/PTS/CTC 전체 교체 (실제 청구항 데이터)
def update_with_new_claims(df, new_df):
    df = df.merge(
        new_df.rename(columns={'CTS': 'CTS_new', 'PTS': 'PTS_new', 'CTC': 'CTC_new'}),
        on='patent_id', how='left'
    )
    for old, new in [('CTS', 'CTS_new'), ('PTS', 'PTS_new'), ('CTC', 'CTC_new')]:
        if old in df.columns and new in df.columns:
            # 새 파일에 있으면 교체, 없으면 기존 값 유지
            df[old] = df[new].combine_first(df[old])
            df.drop(columns=[new], inplace=True)
    return df

print('\npv_L 업데이트...', end=' ', flush=True)
pv_L  = update_with_new_claims(pv_L,  claims_new)
print(f'{len(pv_L):,}건')

print('pv_NL 업데이트...', end=' ', flush=True)
pv_NL = update_with_new_claims(pv_NL, claims_new)
print(f'{len(pv_NL):,}건')

# ── 재저장 ────────────────────────────────────────────────────────
pv_L.to_pickle(f'{DATA_DIR}/pv_L.pkl')
pv_NL.to_pickle(f'{DATA_DIR}/pv_NL.pkl')

print('\npv_L.pkl, pv_NL.pkl 재저장 완료')
print('\n업데이트 후 CTS/PTS/CTC 결측률')
for col in ['CTS', 'PTS', 'CTC']:
    if col in pv_L.columns:
        print(f'  L  {col}: {pv_L[col].isna().mean():.1%}')
        print(f'  NL {col}: {pv_NL[col].isna().mean():.1%}')

1999년 처리 중... 153,588건
2000년 처리 중... 157,596건
2001년 처리 중... 166,158건
2002년 처리 중... 167,400건
2003년 처리 중... 169,077건

2004-2015 통합: 813,819건 unique 특허

pv_L 업데이트... 33,657건
pv_NL 업데이트... 168,285건

pv_L.pkl, pv_NL.pkl 재저장 완료

업데이트 후 CTS/PTS/CTC 결측률
  L  CTS: 0.2%
  NL CTS: 34.6%
  L  PTS: 0.2%
  NL PTS: 34.6%
  L  CTC: 0.2%
  NL CTC: 34.6%


In [8]:
# 처리 완료된 연도 파일 삭제 (디스크 공간 확보)
deleted_total = 0.0
for year in range(1978, 1999):
    path = f'{PATENTSVIEW_DIR}/g_claims_{year}.tsv'
    if os.path.exists(path):
        size = os.path.getsize(path) / 1e9
        os.remove(path)
        deleted_total += size
        print(f'  삭제: g_claims_{year}.tsv  ({size:.2f} GB)')
    else:
        print(f'  없음 (이미 삭제됨): g_claims_{year}.tsv')

print(f'\n총 {deleted_total:.2f} GB 확보')

  삭제: g_claims_1978.tsv  (0.31 GB)
  삭제: g_claims_1979.tsv  (0.23 GB)
  삭제: g_claims_1980.tsv  (0.29 GB)
  삭제: g_claims_1981.tsv  (0.31 GB)
  삭제: g_claims_1982.tsv  (0.28 GB)
  삭제: g_claims_1983.tsv  (0.28 GB)
  삭제: g_claims_1984.tsv  (0.33 GB)
  삭제: g_claims_1985.tsv  (0.36 GB)
  삭제: g_claims_1986.tsv  (0.36 GB)
  삭제: g_claims_1987.tsv  (0.43 GB)
  삭제: g_claims_1988.tsv  (0.41 GB)
  삭제: g_claims_1989.tsv  (0.52 GB)
  삭제: g_claims_1990.tsv  (0.49 GB)
  삭제: g_claims_1991.tsv  (0.54 GB)
  삭제: g_claims_1992.tsv  (0.55 GB)
  삭제: g_claims_1993.tsv  (0.57 GB)
  삭제: g_claims_1994.tsv  (0.61 GB)
  삭제: g_claims_1995.tsv  (0.63 GB)
  삭제: g_claims_1996.tsv  (0.70 GB)
  삭제: g_claims_1997.tsv  (0.72 GB)
  삭제: g_claims_1998.tsv  (0.98 GB)

총 9.92 GB 확보


## Step 6. CS — 패밀리 특허 국가 범위
`g_patent_abroad.tsv` 없음 → `g_pct_data.tsv`로 PCT 이진 플래그 사용

**CS 해석**: 특허가 PCT(국제출원) 경로를 통해 국제적으로 출원되었는지 여부  
- `CS = 1` → PCT 출원 경로 포함 (국제 패밀리 있음)  
- `CS = 0` → 미국 단독 출원

> ⚠️ **제한사항**: 원논문의 CS는 패밀리 국가 수이지만, `g_patent_abroad.tsv` 미보유로 이진 플래그로 대체.  
> `g_patent_abroad.tsv` 다운로드 후 교체 권장.

In [13]:
# g_patent_abroad.tsv 존재 여부 확인
abroad_path = f'{PATENTSVIEW_DIR}/g_patent_aborad.tsv'
pct_path    = f'{PATENTSVIEW_DIR}/g_pct_data.tsv'

import os
print(f'g_patent_aborad.tsv 존재: {os.path.exists(abroad_path)}')
print(f'g_pct_data.tsv 존재:      {os.path.exists(pct_path)}')

g_patent_aborad.tsv 존재: False
g_pct_data.tsv 존재:      True


In [14]:
# g_pct_data.tsv 컬럼 확인 + PCT 플래그 추출
pct_sample = pd.read_csv(pct_path, sep='\t', nrows=5)
print('g_pct_data.tsv 컬럼:', pct_sample.columns.tolist())
print(pct_sample)


g_pct_data.tsv 컬럼: ['patent_id', 'published_or_filed_date', 'pct_371_date', 'pct_102_date', 'filed_country', 'application_kind', 'pct_doc_number', 'pct_doc_type']
   patent_id published_or_filed_date pct_371_date  pct_102_date filed_country  \
0   12512528              2020-10-29   2022-05-06           NaN            WO   
1    9676234              2011-06-17   2013-05-16           NaN            WO   
2    9289251              2012-12-27          NaN           NaN            WO   
3   10954531              2017-12-14          NaN           NaN            WO   
4   10336890              2015-02-18   2016-08-19           NaN            WO   

  application_kind     pct_doc_number     pct_doc_type  
0              NaN  PCT/JP2020/040672  pct_application  
1               00  PCT/JP2011/063937  pct_application  
2                A      WO2012/175871         wo_grant  
3                A      WO2017/213152         wo_grant  
4               00  PCT/JP2015/054359  pct_application  


In [15]:
# PCT patent_id 추출 → CS 이진 플래그
pct_df = pd.read_csv(pct_path, sep='\t', low_memory=False)
pct_df['patent_id'] = pd.to_numeric(pct_df['patent_id'], errors='coerce')
pct_df = pct_df.dropna(subset=['patent_id']).astype({'patent_id': int})

# 특허당 PCT 출원국 수 (CS = PCT 경유 국가 수, 없으면 0)
if 'filed_country' in pct_df.columns:
    cs_df = (
        pct_df.groupby('patent_id')['filed_country'].nunique()
        .reset_index().rename(columns={'filed_country': 'CS'})
    )
else:
    # filed_country 없으면 이진 플래그로
    cs_df = (
        pct_df[['patent_id']].drop_duplicates()
        .assign(CS=1)
    )
    
print(f'PCT 특허: {len(cs_df):,}건')
print(cs_df['CS'].describe())

# 소송특허 커버리지
l_in_pct = len(litigated_set_int & set(cs_df['patent_id']))
print(f'\nL 커버리지: {l_in_pct:,} / {len(litigated_set_int):,} = {l_in_pct/len(litigated_set_int):.1%}')


PCT 특허: 1,142,680건
count   1142680.0000
mean          1.0000
std           0.0000
min           1.0000
25%           1.0000
50%           1.0000
75%           1.0000
max           1.0000
Name: CS, dtype: float64

L 커버리지: 1,321 / 33,657 = 3.9%


## Step 7. [IMPROVED] TCP — 기술집중도 (원고 특허 보유 수)
**TCP**: 소송 원고 기업의 특허 보유 수 (`us_transfer_register_litigation.REGISTER_COUNT`)  
→ 원고명 정규화(법인 접미사 제거) 후 `RGTR_NM`과 매칭

In [17]:
conn_ai = pymysql.connect(
    host=DB_116_HOST, user=DB_USER, password=DB_PW,
    database='AI_Evaluation', charset='utf8mb4'
)
tcp_raw = pd.read_sql(
    """SELECT RGTR_NM, REGISTER_COUNT
       FROM us_transfer_register_litigation
       WHERE RGTR_NM IS NOT NULL""",
    conn_ai
)
print(f'us_transfer_register_litigatiopn: {len(tcp_raw):,}건')
print(tcp_raw.head(3))

us_transfer_register_litigatiopn: 341,367건
                                     RGTR_NM  REGISTER_COUNT
0                                  !OBAC LTD              10
1  "21" INTERNATIONAL HOLDINGS INC, A  OF DE               0
2                          "3D MEDICINE" LLC               0


In [41]:
# ── 기업명 정규화 함수 ────────────────────────────────────────────
_STATE_PAT = re.compile(
    r'\bA\s+(?:CORPORATION\s+)?OF\s+\w{2,4}\.?\b', re.IGNORECASE
)
_AKA_PAT = re.compile(
    r'[,\s]*(?:A/K/A|D/B/A|F/K/A|T/A|AKA|DBA|FKA|'
    r'ALSO\s+KNOWN\s+AS|DOING\s+BUSINESS\s+AS|FORMERLY\s+KNOWN\s+AS|'
    r'TRADING\s+AS)\b.*$',
    re.IGNORECASE
)
_SUFFIX_PAT = re.compile(
    r'\b(INCORPORATED|CORPORATION|COMPANY|LIMITED|ENTERPRISES|'
    r'INC|CORP|LLC|LTD|CO|PLC|LP|LLP|'
    r'GMBH|AG|SA|SL|SRL|BV|NV|AB|AS|KK|'
    r'THE|AND)\b',
    re.IGNORECASE
)

def normalize_company(name):
    if pd.isna(name):
        return None
    s = str(name).upper().strip()
    s = _STATE_PAT.sub(' ', s)         # "A OF OH." 계열 통째 제거
    s = re.sub(r'\.', '', s)           # 마침표 전체 제거 → J.M.→JM, S.L.→SL, L.L.C.→LLC
    s = _AKA_PAT.sub('', s)            # A/K/A, D/B/A, F/K/A 이후 전부 제거 (/ 제거 전에 먼저)
    s = re.sub(r'[^\w\s]', ' ', s)     # 나머지 특수문자 → 공백
    s = _SUFFIX_PAT.sub(' ', s)        # 법인 접미사 제거
    s = re.sub(r'\s+', ' ', s).strip()
    return s if s else None

# TCP 테이블 정규화
tcp_raw['rgtr_norm'] = tcp_raw['RGTR_NM'].apply(normalize_company)

# 동일 기업 중복 시 최대 보유 수 사용
tcp_norm = (
    tcp_raw.dropna(subset=['rgtr_norm'])
    .groupby('rgtr_norm')['REGISTER_COUNT'].max()
    .reset_index().rename(columns={'REGISTER_COUNT': 'TCP'})
)

print(f'정규화 전: {len(tcp_raw):,}건  →  정규화 후 unique: {len(tcp_norm):,}건')
print(tcp_norm.sort_values('TCP', ascending=False).head(10))

정규화 전: 341,367건  →  정규화 후 unique: 304,313건
                              rgtr_norm     TCP
134140  INTERNATIONAL BUSINESS MACHINES  155982
233459              SAMSUNG ELECTRONICS  107418
45350            CANON KABUSHIKI KAISHA   79173
104043                 GENERAL ELECTRIC   47687
218849                         QUALCOMM   46225
250965                             SONY   45390
132858                            INTEL   44667
157486                   LG ELECTRONICS   44327
45466                    CANTOR COLBURN   42903
143678         KABUSHIKI KAISHA TOSHIBA   42110


In [43]:
# ── 원고명 정규화 & TCP 매핑 ──────────────────────────────────────
case_plaintiff = (
    names[names['party_type'].str.strip().str.lower() == 'plaintiff']
    [['case_row_id', 'name']]
    .drop_duplicates('case_row_id')
    .rename(columns={'name': 'plaintiff'})
)
case_plaintiff['plaintiff'] = case_plaintiff['plaintiff'].str.upper()
case_plaintiff['plt_norm'] = case_plaintiff['plaintiff'].apply(normalize_company)

# case_row_id → TCP 매핑
case_tcp = case_plaintiff.merge(
    tcp_norm.rename(columns={'rgtr_norm': 'plt_norm'}),
    on='plt_norm', how='left'
)

# patent → case_row_id → TCP
patent_tcp = (
    lit_patents[['case_row_id', 'patent']]
    .merge(case_tcp[['case_row_id', 'TCP']], on='case_row_id', how='left')
    .groupby('patent')['TCP'].max()
    .reset_index()
)
patent_tcp['patent_id'] = pd.to_numeric(patent_tcp['patent'], errors='coerce')

match_rate = case_tcp['TCP'].notna().mean()
print(f'TCP 매핑 완료: {len(patent_tcp):,}건')
print(f'원고-TCP 매칭률: {match_rate:.1%}')
print(f'TCP notna: {patent_tcp["TCP"].notna().sum():,}건 / {len(patent_tcp):,}건')

TCP 매핑 완료: 38,214건
원고-TCP 매칭률: 63.9%
TCP notna: 28,772건 / 38,214건


In [50]:
# ── Step 8. CD / IC — PatentsView 출원인 데이터 ───────────────────
all_target = litigated_set_int | nl_set_int

print('g_assignee_not_disambiguated.tsv 로딩...', flush=True)
asgn_df = read_pv_chunks(
    f'{PATENTSVIEW_DIR}/g_assignee_not_disambiguated.tsv',
    ['patent_id', 'assignee_sequence', 'rawlocation_id'],
    all_target
)
print(f'  assignee 행 수: {len(asgn_df):,}')

# CD: 특허당 출원인 수
cd_df = (
    asgn_df.groupby('patent_id')['assignee_sequence'].count()
    .reset_index().rename(columns={'assignee_sequence': 'CD'})
)

# IC: 출원인 국가 수 — g_location_not_disambiguated.tsv join (FIE와 동일 패턴)
need_locs = set(asgn_df['rawlocation_id'].dropna().unique())
loc_chunks = []
for chunk in pd.read_csv(
    f'{PATENTSVIEW_DIR}/g_location_not_disambiguated.tsv', sep='\t',
    usecols=['rawlocation_id', 'raw_country'], low_memory=False, chunksize=500_000
):
    loc_chunks.append(chunk[chunk['rawlocation_id'].isin(need_locs)])
loc_df = pd.concat(loc_chunks).drop_duplicates('rawlocation_id')

ic_df = (
    asgn_df.merge(loc_df, on='rawlocation_id', how='left')
    .groupby('patent_id')['raw_country'].nunique()
    .reset_index().rename(columns={'raw_country': 'IC'})
)

print(f'CD: {len(cd_df):,}건')
print(f'IC: {len(ic_df):,}건')
print(cd_df.describe().round(2))
print(ic_df.describe().round(2))

g_assignee_not_disambiguated.tsv 로딩...
  assignee 행 수: 186,902
CD: 180,327건
IC: 180,327건
          patent_id          CD
count   180327.0000 180327.0000
mean   8111609.8900      1.0400
std    2358116.1700      0.2200
min    3930273.0000      1.0000
25%    6186006.5000      1.0000
50%    7912132.0000      1.0000
75%   10091025.0000      1.0000
max   12514120.0000     11.0000
          patent_id          IC
count   180327.0000 180327.0000
mean   8111609.8900      1.0000
std    2358116.1700      0.1100
min    3930273.0000      0.0000
25%    6186006.5000      1.0000
50%    7912132.0000      1.0000
75%   10091025.0000      1.0000
max   12514120.0000      5.0000


In [52]:
def build_feature_matrix(pv_feat, target_type='L'):
    """
    PatentsView 기본 피처(pv_feat)에 Claims(CTS/PTS/CTC), CD, IC를 LEFT JOIN.
    target_type='L'이면 TCP, LI도 추가.
    CS 제외 — g_patent_abroad.tsv 미보유, PCT proxy 커버리지 3.9%로 무의미.
    """
    df = pv_feat.copy()
    df = df.merge(cd_df, on='patent_id', how='left')
    df = df.merge(ic_df, on='patent_id', how='left')

    if target_type == 'L':
        df = df.merge(patent_tcp[['patent_id', 'TCP']], on='patent_id', how='left')
        df = df.merge(patent_li[['patent_id', 'LI1', 'LI3', 'LI5', 'LI7']], on='patent_id', how='left')
    else:
        df['TCP'] = np.nan
        df['LI1'] = 0.0
        df['LI3'] = 0.0
        df['LI5'] = 0.0
        df['LI7'] = 0.0

    return df


print('L 피처 매트릭스 구축...')
df_L = build_feature_matrix(pv_L, target_type='L')
df_L['label'] = 1
print(f'  L: {len(df_L):,}건  / 컬럼: {len(df_L.columns)}개')

print('NL 피처 매트릭스 구축...')
df_NL = build_feature_matrix(pv_NL, target_type='NL')
df_NL['label'] = 0
print(f'  NL: {len(df_NL):,}건  / 컬럼: {len(df_NL.columns)}개')

L 피처 매트릭스 구축...
  L: 33,657건  / 컬럼: 29개
NL 피처 매트릭스 구축...
  NL: 168,285건  / 컬럼: 29개


## Step 9. 전체 피처 통합
L(소송)과 NL(비소송) 각각에 모든 피처를 LEFT JOIN

In [53]:
all_feat_cols = [
    'MCD','SCD','TD_A','TD_B','TD_C','TD_D','TD_E','TD_F','TD_G','TD_H',
    'ABS','CTS','CTC','PTS',
    'PK','SK','PPS','CD','IC','IE','FIE','ET',
    'TCP','LI1','LI3','LI5','LI7'
]

def coverage_report(df, label, cols):
    rows = []
    for c in cols:
        if c in df.columns:
            n = len(df)
            notna = df[c].notna().sum()
            rows.append({'Feature': c, 'Group': label,
                         'N': n, 'N_valid': notna,
                         'Coverage': notna / n})
        else:
            rows.append({'Feature': c, 'Group': label,
                         'N': len(df), 'N_valid': 0, 'Coverage': 0.0})
    return pd.DataFrame(rows)

cov_L  = coverage_report(df_L,  'L (소송)',   all_feat_cols)
cov_NL = coverage_report(df_NL, 'NL (비소송)', all_feat_cols)
cov_all = pd.concat([cov_L, cov_NL])

# 피벗 테이블
cov_pivot = cov_all.pivot(index='Feature', columns='Group', values='Coverage').round(3)
cov_pivot.index = pd.CategoricalIndex(cov_pivot.index, categories=all_feat_cols, ordered=True)
cov_pivot = cov_pivot.sort_index()

print('=== 피처 커버리지 (1.000 = 100%) ===')
print(cov_pivot.to_string())

=== 피처 커버리지 (1.000 = 100%) ===
Group    L (소송)  NL (비소송)
Feature                  
MCD      0.9990    1.0000
SCD      0.9990    1.0000
TD_A     0.9990    1.0000
TD_B     0.9990    1.0000
TD_C     0.9990    1.0000
TD_D     0.9990    1.0000
TD_E     0.9990    1.0000
TD_F     0.9990    1.0000
TD_G     0.9990    1.0000
TD_H     0.9990    1.0000
ABS      0.9990    1.0000
CTS      0.9980    0.6540
CTC      0.9980    0.6540
PTS      0.9980    0.6540
PK       0.9860    0.9130
SK       0.5610    0.5240
PPS      0.1320    0.3890
CD       0.8190    0.9080
IC       0.8190    0.9080
IE       0.9990    1.0000
FIE      0.9990    1.0000
ET       0.9990    1.0000
TCP      0.7640    0.0000
LI1      0.9920    1.0000
LI3      0.9920    1.0000
LI5      0.9920    1.0000
LI7      0.9920    1.0000


In [54]:
# Model 2용: L 특허에 소송 횟수 기반 레이블 추가
lit_count = (
    lit_patents.groupby('patent')['case_row_id'].nunique()
    .reset_index().rename(columns={'case_row_id': 'lit_count'})
)
lit_count['patent_id'] = pd.to_numeric(lit_count['patent'], errors='coerce')

df_L_m2 = df_L.merge(lit_count[['patent_id', 'lit_count']], on='patent_id', how='left')
df_L_m2['label_L1L2'] = (df_L_m2['lit_count'].fillna(1) >=3).astype(int)

print(f'Model 2 데이터: {len(df_L_m2):,}건')
print(f'  L1 (저위험, 1~2회): {(df_L_m2["label_L1L2"]==0).sum():,}건')
print(f'  L2 (고위험, 3회↑): {(df_L_m2["label_L1L2"]==1).sum():,}건')

Model 2 데이터: 33,657건
  L1 (저위험, 1~2회): 26,301건
  L2 (고위험, 3회↑): 7,356건


## Step 10. 피처 커버리지 분석
각 피처별 결측률을 L/NL 그룹별로 확인

In [57]:
# ── 피처 완전성 플래그 — 결측 처리 전에 표시 ─────────────────────
# claims_observed: g_claims_YYYY.tsv에서 실제 청구항 데이터가 있는 특허 = 1
# CTS/PTS/CTC가 0으로 채워질 특허와 실제 청구항 없는 특허를 구분
# 모델 학습 시 1:1 샘플링에서 claims_observed=1 우선 사용

claims_pids_all = set(
    pd.read_pickle(f'{DATA_DIR}/pv_L.pkl')['patent_id']
    .pipe(lambda s: s[s.notna()])
)

# pv_L/pv_NL 에서 CTS가 NaN이 아닌 것 = 실제 claims 데이터 보유
df_L['claims_observed'] = df_L['CTS'].notna().astype(int)
df_NL['claims_observed'] = df_NL['CTS'].notna().astype(int)

print('=== claims_observed 분포 ===')
print(f'L  — 관측: {df_L["claims_observed"].sum():,}건 / 전체: {len(df_L):,}건 ({df_L["claims_observed"].mean():.1%})')
print(f'NL — 관측: {df_NL["claims_observed"].sum():,}건 / 전체: {len(df_NL):,}건 ({df_NL["claims_observed"].mean():.1%})')
print()
print('→ 모델 학습 시 claims_observed=1 먼저 샘플링, 부족하면 0으로 보완')

=== claims_observed 분포 ===
L  — 관측: 33,605건 / 전체: 33,657건 (99.8%)
NL — 관측: 110,081건 / 전체: 168,285건 (65.4%)

→ 모델 학습 시 claims_observed=1 먼저 샘플링, 부족하면 0으로 보완


## Step 11. 결측 처리 전략

| 피처 | 결측 처리 | 근거 |
|------|----------|------|
| PK, SK, PPS | 0 | 인용/우선권 없으면 실제 0 |
| LI1~LI7 | 0 | 이력 없으면 0회 |
| CTS, PTS, CTC | 0 | 2016년 이후 특허 = 청구항 데이터 없음 → 0 대체 |
| MCD, SCD, TD | 0 | IPC 없으면 0 |
| ABS, IE, FIE, ET | 중앙값 | 존재해야 하는 값이므로 중앙값으로 대체 |
| CD, IC | 중앙값 | DB 범위 밖 특허에 대한 결측 |
| TCP | 중앙값 | 원고 매칭 실패 시 중앙값 대체 |

> **CS 제외**: `g_patent_abroad.tsv` 미보유로 PCT proxy(g_pct_data.tsv) 사용 시 L 커버리지 3.9% — 이진 플래그로서도 의미 없어 모델에서 제외.

> **주의**: 결측 처리는 Model1/Model2 학습셋 내에서 각각 fit하고 test에 transform 적용 권장 (Leakage 방지)

In [58]:
FILL_ZERO = ['PK', 'SK', 'PPS', 'LI1', 'LI3', 'LI5', 'LI7',
             'CTS', 'PTS', 'CTC',
             'MCD', 'SCD', 'TD_A', 'TD_B', 'TD_C', 'TD_D', 'TD_E', 'TD_F', 'TD_G', 'TD_H',
             'TCP'] # TCP 미매칭 = 등록 특허 없음(NPE) → 0 대체
FILL_MEDIAN = ['ABS', 'IE', 'FIE', 'ET', 'CD', 'IC']

def impute_features(df, zero_cols, median_cols):
    df = df.copy()
    for c in zero_cols:
        if c in df.columns:
            df[c] = df[c].fillna(0)
    for c in median_cols:
        if c in df.columns:
            m = df[c].median()
            df[c] = df[c].fillna(m)
    return df

df_L_clean = impute_features(df_L, FILL_ZERO, FILL_MEDIAN)
df_NL_clean = impute_features(df_NL, FILL_ZERO, FILL_MEDIAN)
df_L_m2_clean = impute_features(df_L_m2, FILL_ZERO, FILL_MEDIAN)

print('결측 처리 완료')

feat_cols_present = [c for c in all_feat_cols if c in df_L_clean.columns]
remaining_na_L    = df_L_clean[feat_cols_present].isna().sum()
remaining_na_NL   = df_NL_clean[feat_cols_present].isna().sum()
print('\n=== 잔여 결측 (0이어야 정상) ===')
print('L:', remaining_na_L[remaining_na_L > 0].to_dict())
print('NL:', remaining_na_NL[remaining_na_NL > 0].to_dict())

결측 처리 완료

=== 잔여 결측 (0이어야 정상) ===
L: {}
NL: {}


In [59]:
# ── 커버리지 낮은 피처 요약 ────────────────────────────────────────
low_cov = cov_pivot[cov_pivot.min(axis=1) < 0.5].copy()
low_cov['min_coverage'] = low_cov.min(axis=1)
print('=== 커버리지 < 50% 피처 ===')
print(low_cov.sort_values('min_coverage').to_string())

=== 커버리지 < 50% 피처 ===
Group    L (소송)  NL (비소송)  min_coverage
Feature                                
TCP      0.7640    0.0000        0.0000
PPS      0.1320    0.3890        0.1320


#### Step 12. 최종 데이터셋 저장

In [60]:
import pickle

# Model 1 데이터셋: L + NL 통합
df_model1 = pd.concat([df_L_clean, df_NL_clean], ignore_index=True)
df_model1_path = f'{DATA_DIR}/df_model1_features.pkl'
df_model1.to_pickle(df_model1_path)

# Model 2 데이터셋: L만 (LI + label_L1L2 포함)
df_model2_path = f'{DATA_DIR}/df_model2_features.pkl'
df_L_m2_clean.to_pickle(df_model2_path)

# 커버리지 테이블
cov_pivot.to_csv(f'{DATA_DIR}/feature_coverage.csv', encoding='utf-8-sig')

print('=== 저장 완료 ===')
print(f'Model 1: {df_model1_path}  ({len(df_model1):,}건, L={int(df_model1["label"].sum()):,} / NL={(df_model1["label"]==0).sum():,})')
print(f'Model 2: {df_model2_path}  ({len(df_L_m2_clean):,}건, L1={int((df_L_m2_clean["label_L1L2"]==0).sum()):,} / L2={int((df_L_m2_clean["label_L1L2"]==1).sum()):,})')
print(f'Coverage: {DATA_DIR}/feature_coverage.csv')

=== 저장 완료 ===
Model 1: ./data/df_model1_features.pkl  (201,942건, L=33,657 / NL=168,285)
Model 2: ./data/df_model2_features.pkl  (33,657건, L1=26,301 / L2=7,356)
Coverage: ./data/feature_coverage.csv


In [61]:
# ── 최종 피처 기술통계 (Model 1 기준) ─────────────────────────────
feat_cols_present = [c for c in all_feat_cols if c in df_model1.columns]
print(f'\n=== Model 1 피처 기술통계 ({len(feat_cols_present)}개) ===')
print(df_model1[feat_cols_present].describe().round(2).T.to_string())


=== Model 1 피처 기술통계 (27개) ===
           count     mean       std      min      25%      50%       75%         max
MCD  201942.0000   1.3400    0.7000   0.0000   1.0000   1.0000    2.0000     14.0000
SCD  201942.0000   1.5100    0.9500   0.0000   1.0000   1.0000    2.0000     22.0000
TD_A 201942.0000   0.1800    0.3800   0.0000   0.0000   0.0000    0.0000      1.0000
TD_B 201942.0000   0.1900    0.3900   0.0000   0.0000   0.0000    0.0000      1.0000
TD_C 201942.0000   0.1100    0.3200   0.0000   0.0000   0.0000    0.0000      1.0000
TD_D 201942.0000   0.0100    0.1000   0.0000   0.0000   0.0000    0.0000      1.0000
TD_E 201942.0000   0.0300    0.1800   0.0000   0.0000   0.0000    0.0000      1.0000
TD_F 201942.0000   0.0900    0.2800   0.0000   0.0000   0.0000    0.0000      1.0000
TD_G 201942.0000   0.3300    0.4700   0.0000   0.0000   0.0000    1.0000      1.0000
TD_H 201942.0000   0.2800    0.4500   0.0000   0.0000   0.0000    1.0000      1.0000
ABS  201942.0000 114.1600   48.490

In [62]:
# ── L vs NL 피처 평균 비교 ────────────────────────────────────────
comparison = df_model1.groupby('label')[feat_cols_present].mean().T
comparison.columns = ['NL (비소송)', 'L (소송)']
comparison['L/NL 비율'] = (comparison['L (소송)'] / (comparison['NL (비소송)'] + 1e-8)).round(2)
print('=== L vs NL 평균 비교 (비율 높을수록 소송 특허가 더 많음) ===')
print(comparison.sort_values('L/NL 비율', ascending=False).to_string())

=== L vs NL 평균 비교 (비율 높을수록 소송 특허가 더 많음) ===
      NL (비소송)    L (소송)           L/NL 비율
TCP     0.0000 1242.5583 124255833853.2800
LI7     0.0000    1.8751    187507080.5100
LI5     0.0000    1.7172    171724342.7900
LI3     0.0000    1.3889    138891222.5100
LI1     0.0000    0.7610     76096249.3700
SK      7.3923   20.3540            2.7500
CTS    10.0426   24.4688            2.4400
PK     15.7513   35.8049            2.2700
PTS     1.7261    3.8178            2.2100
CTC   634.0883 1352.9453            2.1300
TD_A    0.1618    0.2529            1.5600
TD_E    0.0314    0.0431            1.3700
ABS   113.2711  118.5881            1.0500
IC      1.0041    1.0025            1.0000
ET    998.8645  998.6173            1.0000
FIE     1.0689    1.0576            0.9900
CD      1.0358    1.0163            0.9800
TD_G    0.3315    0.3048            0.9200
IE      2.6182    2.3590            0.9000
MCD     1.3810    1.1537            0.8400
TD_B    0.1918    0.1536            0.8000
SCD     1.

## 요약

| 항목 | 결과 |
|------|------|
| **Model 1 데이터** | L + NL, `df_model1_features.pkl` |
| **Model 2 데이터** | L (L1/L2 레이블), `df_model2_features.pkl` |
| **CTS/PTS/CTC** | g_claims_2016.tsv에서 계산. 2016년 외 특허는 0 대체 |
| **CS** | g_pct_data.tsv PCT 경로 이진 플래그. g_patent_abroad.tsv 다운로드 후 교체 필요 |
| **TCP** | 원고 법인명 정규화 후 REGISTER_COUNT 매핑. 미매칭 → 중앙값 대체 |
| **다음 단계** | 내일 Model 1 (L vs NL) 학습 시작 |